# Практична робота № 4. Відкриті міські дані та індикатори сталості
## Розібраний приклад: якість повітря в Харкові, 2021–2023 (PM2.5)

**Питання дослідження.** Як змінювалася якість повітря в Харкові у 2021–2023 роках за індикатором
«середньорічна концентрація дрібнодисперсних частинок PM2.5» (ISO 37120, ЦСР 11.6.2) і чи можна довіряти
цим даним?

**Дані.** Набір data.gov.ua «Щогодинні концентрації та індекс якості атмосферного повітря для дрібнодисперсного
пилу фракції 2.5 мікронів в населених пунктах України» (розпорядник — ГО «SaveDnipro», ліцензія CC BY).

**Конвеєр:** завантаження → очищення → оцінка якості даних → обчислення індикаторів → візуалізація → експорт → висновки.

### Крок 1. Імпорт бібліотек і налаштування

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Шляхи відносно кореня репозиторію
# (ноутбук лежить у папці notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

YEARS = [2021, 2022, 2023]
# Для свого варіанта змініть ці два значення:
CITY_NAME = "Харків"
REGION_NAME = "Харківська область"

# Нормативи для порівняння, мкг/м³
WHO_ANNUAL = 5       # ВООЗ 2021, середньорічна
WHO_DAILY = 15       # ВООЗ 2021, середньодобова
EU_ANNUAL_2030 = 10  # ЄС з 2030 р., середньорічна
EU_DAILY_2030 = 25   # ЄС з 2030 р., середньодобова

pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)
print("Файли в data/raw:",
      sorted(p.name for p in RAW.glob("*.csv")))

### Крок 2. Довідник населених пунктів: знаходимо ідентифікатор міста

Щогодинні файли містять не назву міста, а числовий ідентифікатор `city_id`. Назви лежать в окремому
довіднику. Це типова структура відкритих даних: **таблиця фактів** (виміри) + **довідник** (опис об'єктів).

In [ ]:
cities = pd.read_csv(RAW / "saveecobot_aqi_cities.csv",
                     encoding="utf-8")
print(cities.shape)
cities.head()

In [ ]:
# Пастка: назви населених пунктів не унікальні!
cities[cities["city_name"] == "Миколаїв"]

In [ ]:
# Шукаємо за назвою, областю й типом і перевіряємо,
# що знайдено рівно один рядок
match = cities[(cities["city_name"] == CITY_NAME)
               & (cities["region_name"] == REGION_NAME)
               & (cities["type_name"] == "місто")]
assert len(match) == 1, (
    f"Знайдено {len(match)} рядків — уточніть умову пошуку")
CITY_ID = int(match["id"].iloc[0])
SLUG = match["url_slug"].iloc[0]  # латинська назва, "kharkiv"
print(f"{CITY_NAME}: city_id = {CITY_ID}, slug = {SLUG}")
match

### Крок 3. Завантаження щогодинних даних

Кожен річний файл містить дані **всіх** населених пунктів (30–50 МБ). Щоб не тримати в пам'яті зайве,
читаємо лише потрібні стовпці з явними типами даних і одразу відбираємо своє місто.

In [ ]:
def read_year(year: int) -> pd.DataFrame:
    name = f"saveecobot_cities_pm25_and_aqi_pm25_{year}.csv"
    df = pd.read_csv(
        RAW / name,
        encoding="utf-8",   # кодування файлу
        sep=",",            # роздільник стовпців
        usecols=["city_id", "pm25", "logged_at"],
        dtype={"city_id": "int32", "pm25": "float64",
               "logged_at": "string"},
    )
    print(f"{year}: усього {len(df):>9,} рядків", end=" → ")
    df = df[df["city_id"] == CITY_ID]
    print(f"для міста {len(df):,}")
    return df

raw = pd.concat([read_year(y) for y in YEARS],
                ignore_index=True)
raw.info()
raw.head()

### Крок 4. Очищення даних

Перевіряємо: типи даних, пропуски, неможливі значення, дублікати, порядок записів.

In [ ]:
df = raw.copy()

# 1) Текст дати-часу → тип datetime (явний формат)
df["logged_at"] = pd.to_datetime(df["logged_at"],
                                 format="%Y-%m-%d %H:%M:%S")

# 2) Пропуски
print("Пропуски за стовпцями:\n", df.isna().sum(), sep="")

# 3) Фізично неможливі значення: < 0 або > 1000 мкг/м³
bad = df[(df["pm25"] < 0) | (df["pm25"] > 1000)]
print("Неможливих значень:", len(bad))

# 4) Дублікати: повні копії рядків і повтор години
print("Повних дублікатів рядків:", df.duplicated().sum())
print("Повторів години:",
      df.duplicated(subset=["logged_at"]).sum())

In [ ]:
# одна година — один вимір; лише можливі значення; за часом
df = (df.drop_duplicates(subset=["logged_at"])
        .loc[lambda d: d["pm25"].between(0, 1000)]
        .sort_values("logged_at")
        .set_index("logged_at"))
print(len(raw), "→", len(df), "рядків після очищення")
df["pm25"].describe().round(1)

### Крок 5. Оцінка якості даних: повнота

**Повнота (completeness)** — частка годин, для яких є вимір. Індикатор, обчислений на неповних даних,
може бути зміщеним: якщо пропущено переважно зимові місяці, середньорічне значення буде заниженим.

In [ ]:
# 1 — у цю годину є вимір, 0 — немає
hours = df["pm25"].resample("h").count()

monthly_completeness = (hours.resample("MS").mean() * 100
                        ).round(1)
completeness_table = (
    monthly_completeness.to_frame("повнота_%")
    .assign(рік=lambda t: t.index.year,
            місяць=lambda t: t.index.month)
    .pivot(index="місяць", columns="рік", values="повнота_%"))
completeness_table

In [ ]:
# Найдовші пропуски: послідовності годин без вимірів
missing = hours.eq(0)
gap_id = (missing != missing.shift()).cumsum()
gaps = (missing[missing].groupby(gap_id[missing])
        .agg(початок=lambda s: s.index.min(),
             тривалість_год="size")
        .sort_values("тривалість_год", ascending=False)
        .reset_index(drop=True))
gaps.head(5)

### Крок 6. Середньодобові значення

За поширеним правилом (зокрема в практиці ЄС) середньодобове значення вважається дійсним,
якщо є щонайменше **18 із 24** годинних вимірів (75 %).

In [ ]:
daily = df["pm25"].resample("D").agg(["mean", "count"])
daily["valid"] = daily["count"] >= 18
# недійсні доби не враховуємо
daily.loc[~daily["valid"], "mean"] = float("nan")
print("Недійсних діб:", (~daily["valid"]).sum(),
      "з", len(daily))
daily.head()

### Крок 7. Обчислення індикаторів

| Індикатор | Методика |
|---|---|
| Середньорічна концентрація PM2.5 | середнє дійсних середньодобових значень за рік (ISO 37120, ЦСР 11.6.2) |
| Повнота даних | частка дійсних діб у році; річне значення вважаємо надійним при ≥ 90 % |
| Дні з перевищенням настанови ВООЗ | кількість діб із середньодобовою концентрацією > 15 мкг/м³ (ВООЗ допускає 3–4 на рік) |
| Дні з перевищенням граничної величини ЄС-2030 | кількість діб > 25 мкг/м³ (допускається до 18 на рік) |

In [ ]:
def who_level(x: float) -> str:
    """Рівень за проміжними цілями ВООЗ (Interim Targets)."""
    levels = [(5, "настанова ВООЗ"), (10, "IT-4"),
              (15, "IT-3"), (25, "IT-2"), (35, "IT-1")]
    for limit, name in levels:
        if x <= limit:
            return name
    return "гірше IT-1"

ind = daily.groupby(daily.index.year).agg(
    середньорічна_pm25=("mean", "mean"),
    дійсних_діб=("valid", "sum"),
    діб_у_році=("valid", "size"),
    діб_понад_15=("mean", lambda s: (s > WHO_DAILY).sum()),
    діб_понад_25=("mean",
                  lambda s: (s > EU_DAILY_2030).sum()),
)
ind["повнота_%"] = (ind["дійсних_діб"] / ind["діб_у_році"]
                    * 100).round(1)
ind["надійно"] = ind["повнота_%"] >= 90
ind["рівень_ВООЗ"] = (ind["середньорічна_pm25"]
                      .apply(who_level))
ind["у_разів_вище_ВООЗ"] = (ind["середньорічна_pm25"]
                            / WHO_ANNUAL).round(1)
ind["середньорічна_pm25"] = ind["середньорічна_pm25"].round(1)
ind.index.name = "рік"
ind

### Крок 8. Візуалізація

In [ ]:
monthly = daily["mean"].resample("MS").mean()
fig, ax = plt.subplots(figsize=(9, 4))
for year in YEARS:
    m = monthly[monthly.index.year == year]
    ax.plot(m.index.month, m.values, marker="o",
            label=str(year))
ax.axhline(WHO_ANNUAL, ls="--", color="gray", lw=1,
           label="ВООЗ, середньорічна (5)")
ax.set_ylim(bottom=0)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Місяць")
ax.set_ylabel("PM2.5, мкг/м³")
ax.set_title(f"{CITY_NAME}: середньомісячна "
             "концентрація PM2.5")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "monthly_pm25.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ind.index.astype(str),
              ind["середньорічна_pm25"], color="#1E2761")
ax.bar_label(bars, fmt="%.1f")
ax.axhline(EU_ANNUAL_2030, ls="--", color="#B8912F",
           label="ЄС з 2030 р. (10)")
ax.axhline(WHO_ANNUAL, ls=":", color="gray",
           label="ВООЗ (5)")
ax.set_ylabel("PM2.5, мкг/м³")
ax.set_title(f"{CITY_NAME}: середньорічна концентрація PM2.5")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "annual_pm25.png", dpi=150)
plt.show()

In [ ]:
# Добовий профіль: середнє для кожної години доби
# в опалювальний (жовтень–квітень) і неопалювальний період
heating = df.index.month.isin([10, 11, 12, 1, 2, 3, 4])
profile = (df.groupby([heating, df.index.hour])["pm25"]
             .mean().unstack(0))
profile.columns = ["неопалювальний", "опалювальний"]
ax = profile.plot(figsize=(8, 4), marker=".")
ax.set_xlabel("Година доби")
ax.set_ylabel("PM2.5, мкг/м³")
ax.set_title(f"{CITY_NAME}: добовий профіль PM2.5, "
             f"{YEARS[0]}–{YEARS[-1]}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / "diurnal_profile.png", dpi=150)
plt.show()

### Крок 9. Експорт результатів

Очищені середньодобові дані знадобляться в Практичній роботі № 5 (дашборд) і № 7 (геоаналіз).

In [ ]:
out = daily.rename(columns={"mean": "pm25_mean",
                            "count": "hours"})
out.to_csv(PROCESSED / f"{SLUG}_pm25_daily.csv")
ind.to_csv(PROCESSED / f"{SLUG}_pm25_indicators.csv")
print("Збережено:",
      sorted(p.name for p in PROCESSED.glob("*.csv")))

### Крок 10. Висновки й обмеження

Висновки записуються у звіт (README.md) з опорою на числа з таблиці індикаторів і графіки.
Обов'язково вкажіть обмеження даних: недорогі сенсори громадян, а не референтні станції; змінна кількість
сенсорів у місті; позначки часу в місцевому часі з переходом на літній час; можливі пропуски
через відключення електроенергії.